# Partial-matching beam search

This notebook demonstrates the replacement beam matcher:

```python
TreePathMatcher(method="beam")
```

The beam is over **valid partial matchings**. A state stores the last matched pair, the accumulated score, and a predecessor pointer. Expanding a state appends a strictly-descendant pair in both trees, so skipped vertices are handled implicitly.

The default heuristic is meant to be usable, not just a placeholder: it indexes labels by subtree preorder intervals, scores positive label-pair buckets, favors rarer informative labels, mildly penalizes large jumps, includes a remaining-height future term, and can add a seeded random tail for exploration.

In [ ]:
from pathlib import Path
import sys


def find_repo_root(start: Path | None = None) -> Path:
    start = Path.cwd() if start is None else start.resolve()
    for candidate in (start, *start.parents):
        if (candidate / "pyproject.toml").exists() and (candidate / "src" / "path_matcher").exists():
            return candidate
    raise RuntimeError("Could not find repository root containing pyproject.toml and src/path_matcher.")


REPO_ROOT = find_repo_root()
SRC = REPO_ROOT / "src"
for path in (str(SRC), str(REPO_ROOT)):
    if path not in sys.path:
        sys.path.insert(0, path)

print("Repo root:", REPO_ROOT)

In [ ]:
import numpy as np

from path_matcher import TreePathMatcher
from path_matcher.tree_data import TreeData


def chain(labels):
    """Create a TreeData path with labels in root-to-leaf order."""
    n = len(labels)
    parent = np.asarray([-1] + list(range(n - 1)), dtype=np.int32)
    return TreeData(parent=parent, label=list(labels), orig_index=np.arange(n, dtype=np.int32))


def eq_or_penalty(a, b):
    return 1.0 if a == b else -0.25


def describe(name, G, H, pairs, score):
    print(f"{name} score:", score)
    print(f"{name} pairs:", pairs)
    print(f"{name} labels:", [(G.label[u], H.label[v]) for u, v in pairs])

## 1. Minimal path example

The exact matcher fills the full dynamic-programming table. The beam matcher searches over partial matching prefixes. On this small example the beam recovers the same path.

In [ ]:
G = chain(list("xAxxBxxC"))
H = chain(list("AByC"))

exact = TreePathMatcher(method="exact", w=eq_or_penalty)
exact_pairs, exact_score = exact.predict(G, H)

beam = TreePathMatcher(
    method="beam",
    w=eq_or_penalty,
    match_predicate=lambda a, b: a == b,
    beam_width=20,
    beam_expansion_width=20,
    beam_symmetric=False,
    seed=0,
)
beam_pairs, beam_score = beam.predict(G, H)

describe("exact", G, H, exact_pairs, exact_score)
describe("beam", G, H, beam_pairs, beam_score)

## 2. Branching trees

A returned beam path must still move downward in both trees. The next matched pair can skip arbitrary descendants, but it cannot move sideways or upward.

In [ ]:
G_branch = TreeData(
    parent=np.asarray([-1, 0, 1, 2, 0, 4, 5], dtype=np.int32),
    label=["root", "A", "B", "C", "A", "X", "C"],
    orig_index=np.arange(7, dtype=np.int32),
)
H_template = chain(["A", "B", "C"])

beam_branch = TreePathMatcher(
    method="beam",
    w=eq_or_penalty,
    match_predicate=lambda a, b: a == b,
    beam_width=10,
    beam_expansion_width=10,
    beam_symmetric=False,
    seed=1,
)
pairs_branch, score_branch = beam_branch.predict(G_branch, H_template)
describe("beam on branching tree", G_branch, H_template, pairs_branch, score_branch)

## 3. Custom beam priority

`beam_priority_fn` receives a `BeamStateContext`. Returning `ctx.default_priority` reproduces the built-in state priority. You can add domain-specific terms, for example length preferences or external scores.

In [ ]:
priority_calls = []


def my_priority(ctx):
    priority_calls.append((ctx.last_u, ctx.last_v, ctx.score, ctx.future_bound))
    # Slightly prefer longer partial matchings when priorities are close.
    return ctx.default_priority + 0.01 * ctx.length

beam_custom_priority = TreePathMatcher(
    method="beam",
    beam_priority_fn=my_priority,
    beam_width=4,
    beam_expansion_width=4,
    beam_symmetric=False,
    seed=0,
)
pairs_priority, score_priority = beam_custom_priority.predict(chain(list("ABC")), chain(list("ABC")))

print("score:", score_priority)
print("pairs:", pairs_priority)
print("number of priority calls:", len(priority_calls))
print("first priority call:", priority_calls[0])

## 4. Custom expansion rule

`beam_expansion_fn` receives a `BeamExpansionContext`. It can return either `(u, v)` pairs, which are scored by `w`, or `(u, v, score)` triples, which use the supplied score. The implementation validates that returned pairs are feasible strict descendants.

In [ ]:
def expansion(ctx):
    # This toy rule only proposes one scored pair from the sentinel state.
    if ctx.last_u == -1 and ctx.last_v == -1:
        return [(2, 1, 5.0)]
    return []

beam_custom_expansion = TreePathMatcher(
    method="beam",
    beam_expansion_fn=expansion,
    beam_width=5,
    beam_expansion_width=5,
    beam_symmetric=False,
)
pairs_expansion, score_expansion = beam_custom_expansion.predict(chain(["A", "B", "C"]), chain(["X", "Y"]))

print("score:", score_expansion)
print("pairs:", pairs_expansion)

## 5. Optional far-lookahead sketching

With a very narrow beam, a locally attractive prefix can crowd out a prefix whose payoff is mostly below the next few descendants. Setting `beam_lookahead=True` adds a preprocessing pass: for every node, the matcher builds capped, depth-discounted sketches of (i) labels in its descendant subtree and (ii) fixed-length downward label chunks. During search, the default candidate and frontier priorities receive an extra score based on sketch compatibility below the current terminal pair. This is intentionally heuristic, not an admissible A* bound.


In [ ]:
G_delayed = TreeData(
    parent=np.asarray([-1, 0, 0, 2, 3, 4], dtype=np.int32),
    label=["root", "A", "g", "A", "B", "C"],
    orig_index=np.arange(6, dtype=np.int32),
)
H_delayed = TreeData(
    parent=np.asarray([-1, 0, 0, 2, 3, 4], dtype=np.int32),
    label=["other", "A", "h", "A", "B", "C"],
    orig_index=np.arange(6, dtype=np.int32),
)

narrow = TreePathMatcher(
    method="beam",
    beam_width=1,
    beam_expansion_width=1,
    beam_random_fraction=0.0,
    seed=0,
)
lookahead = TreePathMatcher(
    method="beam",
    beam_width=1,
    beam_expansion_width=1,
    beam_random_fraction=0.0,
    beam_lookahead=True,
    seed=0,
)

narrow_pairs, narrow_score = narrow.predict(G_delayed, H_delayed)
look_pairs, look_score = lookahead.predict(G_delayed, H_delayed)

describe("narrow beam", G_delayed, H_delayed, narrow_pairs, narrow_score)
describe("lookahead beam", G_delayed, H_delayed, look_pairs, look_score)


## 6. Main knobs

Useful beam-search parameters:

- `beam_width`: number of partial matchings retained after each layer.
- `beam_expansion_width`: maximum number of extension pairs proposed per live state.
- `beam_symmetric`: run the search in both orientations and keep the better score. This roughly doubles work.
- `beam_random_fraction` and `beam_n_restarts`: seeded stochastic exploration.
- `beam_candidate_heuristic_fn`: callable for local candidate ranking.
- `beam_priority_fn`: callable for frontier ranking.
- `beam_expansion_fn`: full custom expansion rule.
- `beam_lookahead`: precompute descendant-label and downward path-chunk sketches and use their overlap as a far-ahead ranking feature.
- `beam_lookahead_weight`, `beam_lookahead_sketch_size`, and `beam_lookahead_depth_discount`: control the strength, memory cap, and depth decay of lookahead.
- `beam_lookahead_chunk_size`, `beam_lookahead_label_weight`, and `beam_lookahead_chunk_weight`: control the fixed-length chunk component and its blend with ordinary descendant-label overlap.
